# Generating a Training Configuration File

Welcome! In this notebook, we are going to create a **Configuration File** (or "config" for short). 

Think of a config file as the **recipe** or **blueprint** for our Machine Learning model. Before we can train an AI to find earthquakes, we need to tell it exactly *how* to train. We need to answer questions like:
- Where is the data?
- How much data should it look at at one time?
- How many times should it review the data to learn?
- Should it use the regular computer processor (CPU) or the super-fast graphics card (GPU)?

Instead of writing these answers directly into our training code, we save them in a clean, organized `JSON` file. This makes it very easy to tweak our recipe later without breaking the code!

In [4]:
import json
import os

## The Recipe: Step-by-Step

Let's build our recipe dictionary. We will divide it into four main sections:

### 1. Peak Detection
When the AI predicts a wave, it draws a probability curve. We need to tell it how tall a "spike" in probability needs to be before we officially call it an earthquake. We set the minimum `height` to 0.5 (50% confidence), and the `distance` to 100 (spikes must be at least 100 samples apart so we don't count the same earthquake twice).

### 2. Data Settings
Here we point the model to our `final_curated_seisbench_data`. We also tell it that our data is recorded at `200` Hz (samples per second).
- `window_len: 1001`: The AI will look at exactly 1001 samples (about 5 seconds) at a time.

### 3. Training Settings
This is how the AI "studies":
- **Batch Size (4):** The AI will look at 4 earthquake examples at a time before updating its brain (like looking at 4 flashcards before checking the answers).
- **Learning Rate (0.01):** How big of a step the AI takes when it learns. Too small, and it learns too slowly. Too big, and it misses the answer.
- **Epochs (50):** The AI will read through the *entire* dataset 50 times.
- **Patience (5):** If the AI stops improving for 5 epochs in a row, it will give up early so it doesn't waste our time.
- **Loss Weights:** We give more "penalty" or "weight" to missing S-waves (0.59) and P-waves (0.4) than missing empty noise (0.01). We really want it to find those waves!

### 4. Device
We tell it to use the GPU (`use_cuda: True`) to train lightning fast.

In [5]:
config = {
    "peak_detection": {
        "sampling_rate": 200,
        "height": 0.5,
        "distance": 100
    },
    "data": {
        "dataset_name": "final_curated_seisbench_data",
        "sampling_rate": 200,
        "window_len": 1001,      # Our traces are exactly 1001 samples long (5 seconds)
        "samples_before": 1000,  # Arrival is centered at sample 1000
        "windowlen_large": 2000,
        "sample_fraction": 1.0
    },
    "training": {
        "batch_size": 4,
        "num_workers": 2,
        "learning_rate": 0.01,
        "epochs": 50,
        "patience": 5,
        "loss_weights": [
            0.01,  
            0.4,   
            0.59   
        ],
        "optimization": {
            "mixed_precision": True,
            "gradient_accumulation_steps": 1,
            "pin_memory": True,
            "prefetch_factor": 2,
            "persistent_workers": True
        }
    },
    "device": {
        "use_cuda": True,
        "device_id": 0
    }
}

## Saving the Configuration

Finally, we use Python's built-in `json` library to write this dictionary into a physical file on our hard drive. 

We use `indent=4` to make the file pretty and easy for a human to read.

In [6]:
output_file = "icequake_train_config.json"

# Write the dictionary to a JSON file
with open(output_file, 'w') as f:
    json.dump(config, f, indent=4)
    
print(f"Training configuration successfully generated: {output_file}")

Training configuration successfully generated: icequake_train_config.json


---
## Conclusion

And that's it! If you look in your folder, you will now see a file named `icequake_train_config.json`. 

When we write our actual model training script, it will simply read this file, look at the recipe, and start training automatically. If we ever want to change the `batch_size` or run it for 100 `epochs` instead of 50, we just change the numbers in the JSON file without touching the complicated training code!